# FloraSense C3 Experiment 2 — fixed-split A/B, hosted Colab T4

Upload both `c3_features_v2_20261022.zip` and `c3_experiment2_support.zip` to
**My Drive / FloraSense_C3 / experiment2**. Use Runtime → Change runtime type →
T4 GPU. Run cells in order. No local runtime, new splits or Experiment 1 weights.
Class 1 is Not Acceptable (root) / One Side (leaf: branch origins, not foliage).


In [ ]:
# 1. Mount Drive
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
DRIVE_DIR = Path("/content/drive/MyDrive/FloraSense_C3/experiment2")


In [ ]:
# 2. Validate archives and extract into a fresh runtime directory
import hashlib, json, shutil, stat, zipfile
from datetime import datetime, timezone
def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream,"sha256").hexdigest()
DATA_SHA = "9c505bcbf13b130722da919502238c0a7d4c0e7449d88b73f8e70d3c22098ae4"
SUPPORT_SHA = "3eb95ca4b4266d4b980764e2d3d3203200fa35b1db4773bd8847e09404a3277b"
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
WORK = Path("/content") / ("c3_experiment2_" + stamp)
WORK.mkdir(exist_ok=False)
DATA_ROOT, CODE_ROOT = WORK / "dataset", WORK / "support"
for name, expected, destination in [
    ("c3_features_v2_20261022.zip",DATA_SHA,DATA_ROOT),
    ("c3_experiment2_support.zip",SUPPORT_SHA,CODE_ROOT)]:
    local = WORK / name
    shutil.copyfile(DRIVE_DIR / name,local)
    assert sha256(local)==expected, "Wrong or incomplete archive: " + name
    with zipfile.ZipFile(local) as archive:
        names=archive.namelist()
        assert len(names)==len(set(names))
        for member in archive.infolist():
            path=(destination/member.filename).resolve()
            assert path.is_relative_to(destination.resolve())
            assert ":" not in member.filename and "\\" not in member.filename
            assert not stat.S_ISLNK(member.external_attr >> 16)
        archive.extractall(destination)
print("Archives verified and extracted")


In [ ]:
# 3. Dependencies and hosted GPU (keep Colab's matched CUDA torch/torchvision)
import sys, subprocess, platform
assert platform.system() != "Windows", "Hosted Colab only"
subprocess.check_call([sys.executable,"-m","pip","install","-q","scikit-learn==1.7.2","pandas>=2.2,<3","Pillow>=10","matplotlib>=3.8"])
import torch
assert torch.cuda.is_available(), "Select a T4 GPU runtime"
print("GPU",torch.cuda.get_device_name(0),"torch",torch.__version__)
sys.path.insert(0,str(CODE_ROOT))
from services.c3_quality_assessment.experiment2_training import validate_dataset, run_architecture, compare_results


In [ ]:
# 4. Validate all V2 data and six saved partitions; no model construction
release, manifests, splits = validate_dataset(DATA_ROOT)
print(release["dataset_version"],release["workbook_sha256"])
for task in ("root","leaf"):
    print(task,manifests[task].label.value_counts().to_dict())
    for fold,frame in enumerate(splits[task],1):
        print("fold",fold,frame.groupby(["split","label"]).size().to_dict())
RUN_ROOT = DRIVE_DIR / "runs" / stamp
print("Output:",RUN_ROOT)


## Training configuration

Both approaches load the identical exported train/validation/test rows. Neither
outer nor inner folds are generated. Plant_ID namespaces Sample_ID; varieties
may appear across partitions as in Experiment 1. This is not held-out-variety
evaluation. Root and leaf remain independent. No Plant_ID feature is supplied.

Same original + two fixed augmentation tensors: EXIF orientation, fit to 208,
center on gray 224 canvas, ImageNet normalization, flip, ±4° rotation and mild
color jitter. No crops; junctions must remain visible. Training-only class
weights; seed 42 plus fold; AdamW, batch 16, head LR .001, layer4 LR .00001,
weight decay .01, up to 40 epochs, patience 6. B starts layer4 adaptation at
epoch 1 (no warm-up); both keep BatchNorm running statistics frozen. Validation
weighted loss selects checkpoints. Test partitions are evaluated only afterward.

A and B are initialized independently from ImageNet, never from a full-data
checkpoint. Review results jointly after both complete; do not tune B based on
A outer-test errors. Fold checkpoints are saved to Drive; no deployment refit or
test-selected best fold is produced. Rerunning an existing arm is refused.


In [ ]:
# 5. Run architecture A: frozen backbone + head, separate root/leaf models
run_architecture(DATA_ROOT,RUN_ROOT,"A")


In [ ]:
# 6. Run architecture B: layer4 + head, exact same saved partitions/images
run_architecture(DATA_ROOT,RUN_ROOT,"B")


In [ ]:
# 7. Compare only after both arms complete
import pandas as pd
compare_results(RUN_ROOT)
display(pd.read_csv(RUN_ROOT/"model_comparison.csv"))
print((RUN_ROOT/"comparison_report.md").read_text())
from IPython.display import display, Image as DisplayImage
for architecture in ("A","B"):
    for task in ("root","leaf"):
        display(DisplayImage(filename=str(RUN_ROOT/architecture/task/"confusion_matrix.png")))


In [ ]:
# 8. Export checkpoints, OOF predictions and reports
from google.colab import files
archive=shutil.make_archive("/content/c3_experiment2_models","zip",root_dir=RUN_ROOT)
backup=RUN_ROOT.parent/(stamp+"_models.zip")
shutil.copyfile(archive,backup)
print("Drive backup:",backup)
files.download(archive)


## Limits and inference handoff

Checkpoints: `{A,B}/{root,leaf}/fold_{1,2,3}.pth`, each with full ResNet18 state,
class mapping, preprocessing and dataset/split provenance. These are fold models;
no full-data final_model.pth is selected by outer scores. Original Experiment 1
artifacts are not used or modified.

The state-dict structure matches C3 inference, but the current general-quality
loader uses a hardcoded path and a different resize transform. Do not replace
its checkpoint. Future feature inference must use the task-specific path/classes
and exact metadata preprocessing (EXIF + contain/padding + normalization).

Minority support is small; corrected P07 has no unacceptable roots, P05 no One
Side leaves and P08 no data. Missing-class per-plant metrics are unavailable.
Previously reviewed outer data is a comparative benchmark, not a fresh external
test. Branch qualification/visibility issues remain; no labels are altered or
test examples excluded. Colab disconnects preserve Drive checkpoints; resume is
not implemented, so start a new run directory after interruption.
